# 7: Fair Concentration Comparison — GCN vs GAT

Notebook 6 compared GCN and GAT using **Gini coefficient** as a concentration metric. This is unfair because Gini conflates:

1. **How focused the strong weights are** (what we actually want to measure)
2. **How many entries are exactly zero** (representational artefact)

GCN rows are hard-thresholded Pearson correlations (~15-30 non-zeros out of 88), while GAT softmax rows have **all 88 non-zero** (but many are tiny). Gini inflates for GCN purely because of its zeros.

### Fair approach

For each window and node, restrict **both** models to the **shared Pearson-masked edge set**, renormalize rows to sum to 1, then apply scale-free concentration metrics:

- **Effective neighbors** `N_eff = exp(-Σ p_i log p_i)` — equivalent number of strong neighbors
- **Top-k mass** — fraction of attention in the top-k neighbors (k=3, 5, 10)
- **Herfindahl index** `H = Σ p_i²` — concentration dominance measure

All three are scale-free and unaffected by the presence of zeros. Same two models as notebook 6: GCN rolling th0.4 and GAT 4c th0.4.

## 0. Setup

In [7]:
import os, sys

# --- Toggle: force Google Drive path even when running locally ---
USE_DRIVE = True   # set False for local FINAL_RESULTS folder

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        get_ipython().system('git clone https://github.com/adam-909/4yp.git /content/repo')
    else:
        get_ipython().system('cd /content/repo && git pull')
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    if USE_DRIVE:
        # Try known local Drive mount points
        drive_candidates = [
            "/mnt/g/My Drive/FINAL_RESULTS",
            "/mnt/g/Shared drives/FINAL_RESULTS",
            os.path.expanduser("~/gdrive/FINAL_RESULTS"),       # rclone default
            os.path.expanduser("~/GoogleDrive/FINAL_RESULTS"),  # alternative
        ]
        RESULTS_BASE = next((p for p in drive_candidates if os.path.exists(p)), None)
        if RESULTS_BASE is None:
            raise FileNotFoundError(
                "USE_DRIVE=True but no Google Drive mount found. Checked:\n  "
                + "\n  ".join(drive_candidates)
                + "\nInstall Google Drive for Desktop (Windows) or rclone (Linux), "
                  "or set USE_DRIVE=False to use local FINAL_RESULTS."
            )
    else:
        RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")
print(f"Results base: {RESULTS_BASE}")


Working directory: /home/adam/new4YP/4YP-main
Results base: /home/adam/gdrive/FINAL_RESULTS


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from gml.experiment_utils import load_experiment_results
from settings.default import ALL_TICKERS, BBG_SECTORS

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.facecolor": "white",
})

# Match notebook 6 colour scheme
GCN_COLOR = "#1f77b4"
GAT_COLOR = "#d62728"

print("Imports ready")

Imports ready


## 1. Load Models

In [ ]:
# Same two models as notebook 6
GCN_EXPERIMENT = "3_GCN_rolling_pearson"
GCN_CONFIG = "lb20_th0.4_equity"
GCN_SEED = 42

GAT_EXPERIMENT = "4c_GATv2_rolling_pearson_th0.4_scFalse_resTrue"
GAT_CONFIG = "lb20_th0.4_equity"
GAT_SEED = 40

print(f"Loading GCN: {GCN_EXPERIMENT}/{GCN_CONFIG}/seed_{GCN_SEED}")
gcn_results = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)

print(f"Loading GAT: {GAT_EXPERIMENT}/{GAT_CONFIG}/seed_{GAT_SEED}")
gat_results = load_experiment_results(GAT_EXPERIMENT, GAT_SEED, config_name=GAT_CONFIG, base_dir=RESULTS_BASE)

gcn_adj = gcn_results["adjacency"]         # (W, 88, 88) — Pearson-thresholded
gat_attn = gat_results["attention_weights"] # (W, 4, 88, 88) — softmax over Pearson-masked
gat_adj = gat_results["adjacency"]          # (W, 88, 88) — same Pearson mask

gcn_dates = pd.to_datetime(gcn_results["test_dates"])
gat_dates = pd.to_datetime(gat_results["test_dates"])

# Head-averaged attention
gat_attn_avg = gat_attn.mean(axis=1)  # (W, 88, 88)

# Ticker / sector setup
tickers = sorted(ALL_TICKERS)
sectors = [BBG_SECTORS.get(t, "Unknown") for t in tickers]
unique_sectors = sorted(set(sectors))
sector_indices = {}
for i, (t, s) in enumerate(zip(tickers, sectors)):
    sector_indices.setdefault(s, []).append(i)

print(f"\nGCN adj: {gcn_adj.shape}")
print(f"GAT attn avg: {gat_attn_avg.shape}")
print(f"GCN windows: {len(gcn_dates)}, GAT windows: {len(gat_dates)}")
print(f"Tickers: {len(tickers)}, Sectors: {len(unique_sectors)}")

Loading GCN: 3_GCN_rolling_pearson/lb20_th0.4_equity/seed_42


In [ ]:
# --- VIX for regime breakdown ---
import yfinance as yf

vix = yf.download("^VIX", start="2017-01-01", end="2023-09-01")["Close"].squeeze()
vix.index = pd.to_datetime(vix.index)

vix_gcn = vix.reindex(gcn_dates, method="ffill")
vix_gat = vix.reindex(gat_dates, method="ffill")

vix_q25 = vix_gcn.quantile(0.25)
vix_q75 = vix_gcn.quantile(0.75)

vix_regime_gcn = pd.Series("Mid", index=gcn_dates)
vix_regime_gcn[vix_gcn > vix_q75] = "High"
vix_regime_gcn[vix_gcn < vix_q25] = "Low"

vix_regime_gat = pd.Series("Mid", index=gat_dates)
vix_regime_gat[vix_gat > vix_q75] = "High"
vix_regime_gat[vix_gat < vix_q25] = "Low"

print(f"VIX quartiles: Q25={vix_q25:.1f}, Q75={vix_q75:.1f}")
print(f"GCN regime counts: {vix_regime_gcn.value_counts().to_dict()}")

## 2. Restriction + Renormalization

For each window `t` and node `i`, restrict both models' row `i` to the Pearson-masked edge set (with self-loops) and renormalize to sum to 1. This puts both models on the same support.

In [ ]:
# Use the GAT adjacency (Pearson mask) as the shared support.
# This is the exact mask used during GAT training, and it matches the GCN's Pearson graph.
# Add self-loops (consistent with how GAT layer processes the adjacency).

n_win_gcn = gcn_adj.shape[0]
n_win_gat = gat_adj.shape[0]
n_win = min(n_win_gcn, n_win_gat)  # align lengths if small mismatch
N = gcn_adj.shape[-1]

# Build shared mask: Pearson edges + self-loops, boolean
eye = np.eye(N, dtype=bool)
shared_mask = (gat_adj[:n_win] > 0) | eye[None, :, :]  # (W, N, N) boolean

print(f"Shared mask: {shared_mask.shape}")
print(f"Mean edges per row (incl self-loop): {shared_mask.sum(axis=-1).mean():.2f}")
print(f"Min / max edges per row: {shared_mask.sum(axis=-1).min()} / {shared_mask.sum(axis=-1).max()}")

# Restrict + renormalize for each model
def restrict_and_renormalize(weights, mask):
    """Zero out entries outside mask and renormalize rows to sum to 1.

    weights: (W, N, N) float — raw edge weights
    mask: (W, N, N) bool — shared support
    Returns: (W, N, N) probability distribution per row, zeros outside mask.
    """
    restricted = weights * mask
    row_sums = restricted.sum(axis=-1, keepdims=True)  # (W, N, 1)
    # Avoid division by zero (rows with zero mass stay zero)
    row_sums = np.where(row_sums > 0, row_sums, 1.0)
    return restricted / row_sums

# For GCN, edge weights are Pearson correlations. Self-loops are 0 in gcn_adj, so add them.
gcn_weights = gcn_adj[:n_win].copy()
for t in range(n_win):
    np.fill_diagonal(gcn_weights[t], 1.0)  # self-loop weight = 1 (Pearson with itself)

# For GAT, attention already sums to 1 over the Pearson-masked support (with self-loops).
gat_weights = gat_attn_avg[:n_win].copy()

gcn_prob = restrict_and_renormalize(gcn_weights, shared_mask)
gat_prob = restrict_and_renormalize(gat_weights, shared_mask)

# Sanity check: rows should sum to ~1
print(f"\nGCN row sums: mean={gcn_prob.sum(axis=-1).mean():.4f}, std={gcn_prob.sum(axis=-1).std():.4f}")
print(f"GAT row sums: mean={gat_prob.sum(axis=-1).mean():.4f}, std={gat_prob.sum(axis=-1).std():.4f}")

## 3. Concentration Metrics — Summary Comparison

In [ ]:
def compute_concentration_metrics(prob):
    """Compute per-row concentration metrics.

    prob: (W, N, N) where prob[t, i, :] is a probability distribution (sums to 1).
    Returns dict of arrays, each shape (W, N) — per window per node.
    """
    # Effective neighbors: N_eff = exp(entropy)
    with np.errstate(divide="ignore", invalid="ignore"):
        log_p = np.where(prob > 0, np.log(prob), 0.0)
    entropy = -np.sum(prob * log_p, axis=-1)  # (W, N)
    n_eff = np.exp(entropy)

    # Top-k mass
    def top_k_mass(p, k):
        # Sort descending along last axis, sum top k
        sorted_desc = -np.sort(-p, axis=-1)
        return sorted_desc[..., :k].sum(axis=-1)

    top3 = top_k_mass(prob, 3)
    top5 = top_k_mass(prob, 5)
    top10 = top_k_mass(prob, 10)

    # Herfindahl
    herfindahl = np.sum(prob ** 2, axis=-1)

    return {
        "n_eff": n_eff,
        "top3_mass": top3,
        "top5_mass": top5,
        "top10_mass": top10,
        "herfindahl": herfindahl,
        "entropy": entropy,
    }

gcn_metrics = compute_concentration_metrics(gcn_prob)
gat_metrics = compute_concentration_metrics(gat_prob)

print("Per-node per-window concentration metrics computed.")
print(f"Sample shape (n_eff): {gcn_metrics['n_eff'].shape}")

In [ ]:
# Summary table + paired t-tests
rows = []
metric_names = ["n_eff", "top3_mass", "top5_mass", "top10_mass", "herfindahl", "entropy"]
metric_labels = {
    "n_eff": "Effective neighbors (N_eff)",
    "top3_mass": "Top-3 mass",
    "top5_mass": "Top-5 mass",
    "top10_mass": "Top-10 mass",
    "herfindahl": "Herfindahl H",
    "entropy": "Entropy (nats)",
}

for m in metric_names:
    g_vals = gcn_metrics[m].flatten()
    a_vals = gat_metrics[m].flatten()
    # Paired t-test (each observation is a window-node pair)
    t_stat, p_val = stats.ttest_rel(g_vals, a_vals)
    # Cohen's d for paired samples
    diff = g_vals - a_vals
    cohens_d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
    rows.append({
        "Metric": metric_labels[m],
        "GCN mean": g_vals.mean(),
        "GCN std": g_vals.std(),
        "GAT mean": a_vals.mean(),
        "GAT std": a_vals.std(),
        "Diff (GCN-GAT)": g_vals.mean() - a_vals.mean(),
        "t-stat": t_stat,
        "p-value": p_val,
        "Cohen's d": cohens_d,
    })

summary_df = pd.DataFrame(rows).set_index("Metric")
print("Concentration metrics — paired comparison over all (window, node) pairs:\n")
print(summary_df.to_string(float_format="%.4f"))
print(f"\nN = {gcn_metrics['n_eff'].size:,} observations per metric")
print("\nInterpretation:")
print("  - If GCN is genuinely MORE concentrated than GAT:")
print("    - GCN N_eff < GAT N_eff")
print("    - GCN top-k mass > GAT top-k mass")
print("    - GCN Herfindahl > GAT Herfindahl")
print("    - GCN entropy < GAT entropy")

## 4. Distribution Comparison

In [ ]:
# Overlaid histograms + CDFs for the 3 headline metrics
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

headline = [("n_eff", "Effective neighbors (N_eff)"),
            ("top5_mass", "Top-5 mass"),
            ("herfindahl", "Herfindahl H")]

for col, (m, label) in enumerate(headline):
    g_vals = gcn_metrics[m].flatten()
    a_vals = gat_metrics[m].flatten()

    # Histogram (row 0)
    ax = axes[0, col]
    bins = np.linspace(min(g_vals.min(), a_vals.min()),
                       max(g_vals.max(), a_vals.max()), 60)
    ax.hist(g_vals, bins=bins, alpha=0.5, label="GCN", color=GCN_COLOR, density=True)
    ax.hist(a_vals, bins=bins, alpha=0.5, label="GAT", color=GAT_COLOR, density=True)
    ax.axvline(g_vals.mean(), color=GCN_COLOR, ls="--", lw=1)
    ax.axvline(a_vals.mean(), color=GAT_COLOR, ls="--", lw=1)
    ax.set_xlabel(label)
    ax.set_ylabel("Density")
    ax.legend()
    ax.set_title(f"Histogram: {label}")

    # CDF (row 1)
    ax = axes[1, col]
    g_sorted = np.sort(g_vals)
    a_sorted = np.sort(a_vals)
    g_cdf = np.arange(len(g_sorted)) / len(g_sorted)
    a_cdf = np.arange(len(a_sorted)) / len(a_sorted)
    ax.plot(g_sorted, g_cdf, color=GCN_COLOR, lw=1.5, label="GCN")
    ax.plot(a_sorted, a_cdf, color=GAT_COLOR, lw=1.5, label="GAT")
    ax.set_xlabel(label)
    ax.set_ylabel("CDF")
    ax.legend()
    ax.set_title(f"CDF: {label}")

plt.suptitle("Concentration metric distributions (per window, per node)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# KS tests for distributional difference
print("\nKolmogorov-Smirnov tests (full distribution difference):")
for m, label in headline:
    ks_stat, ks_p = stats.ks_2samp(gcn_metrics[m].flatten(), gat_metrics[m].flatten())
    print(f"  {label:30s}: KS = {ks_stat:.4f}, p = {ks_p:.2e}")

## 5. Time-Series Breakdown

In [ ]:
# Average across nodes per window → time series
gcn_neff_t = gcn_metrics["n_eff"].mean(axis=-1)  # (W,)
gat_neff_t = gat_metrics["n_eff"].mean(axis=-1)
gcn_top5_t = gcn_metrics["top5_mass"].mean(axis=-1)
gat_top5_t = gat_metrics["top5_mass"].mean(axis=-1)
gcn_herf_t = gcn_metrics["herfindahl"].mean(axis=-1)
gat_herf_t = gat_metrics["herfindahl"].mean(axis=-1)

aligned_dates = gcn_dates[:n_win]
vix_aligned = vix_gcn.reindex(aligned_dates, method="ffill").values

fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)

# Panel 1: N_eff
ax = axes[0]
ax.plot(aligned_dates, pd.Series(gcn_neff_t).rolling(20).mean(),
        color=GCN_COLOR, lw=1.5, label="GCN")
ax.plot(aligned_dates, pd.Series(gat_neff_t).rolling(20).mean(),
        color=GAT_COLOR, lw=1.5, label="GAT")
ax.set_ylabel("Effective neighbors (20d rolling)")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(aligned_dates, vix_aligned, color="gray", lw=0.5, alpha=0.5)
ax2.set_ylabel("VIX", color="gray")
ax2.grid(False)
ax.set_title("Effective neighbors over time (higher = more uniform)")

# Panel 2: Top-5 mass
ax = axes[1]
ax.plot(aligned_dates, pd.Series(gcn_top5_t).rolling(20).mean(),
        color=GCN_COLOR, lw=1.5, label="GCN")
ax.plot(aligned_dates, pd.Series(gat_top5_t).rolling(20).mean(),
        color=GAT_COLOR, lw=1.5, label="GAT")
ax.set_ylabel("Top-5 mass (20d rolling)")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(aligned_dates, vix_aligned, color="gray", lw=0.5, alpha=0.5)
ax2.set_ylabel("VIX", color="gray")
ax2.grid(False)
ax.set_title("Top-5 mass over time (higher = more concentrated)")

# Panel 3: Herfindahl
ax = axes[2]
ax.plot(aligned_dates, pd.Series(gcn_herf_t).rolling(20).mean(),
        color=GCN_COLOR, lw=1.5, label="GCN")
ax.plot(aligned_dates, pd.Series(gat_herf_t).rolling(20).mean(),
        color=GAT_COLOR, lw=1.5, label="GAT")
ax.set_ylabel("Herfindahl (20d rolling)")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(aligned_dates, vix_aligned, color="gray", lw=0.5, alpha=0.5)
ax2.set_ylabel("VIX", color="gray")
ax2.grid(False)
ax.set_title("Herfindahl over time (higher = more concentrated)")
ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

# Correlation of concentration with VIX
print("Correlation of concentration metrics with VIX:")
for name, g_series, a_series in [
    ("N_eff",       gcn_neff_t, gat_neff_t),
    ("Top-5 mass",  gcn_top5_t, gat_top5_t),
    ("Herfindahl",  gcn_herf_t, gat_herf_t),
]:
    valid = ~np.isnan(vix_aligned)
    r_g, p_g = stats.pearsonr(g_series[valid], vix_aligned[valid])
    r_a, p_a = stats.pearsonr(a_series[valid], vix_aligned[valid])
    print(f"  {name:15s}: GCN r={r_g:+.3f} (p={p_g:.2e}), GAT r={r_a:+.3f} (p={p_a:.2e})")

## 6. Regime Breakdown (VIX-based)

In [ ]:
# Per-regime mean metrics
regime_series = vix_regime_gcn.reindex(aligned_dates).values  # (W,)

regime_rows = []
for regime in ["High", "Mid", "Low"]:
    mask_t = (regime_series == regime)
    n_days = mask_t.sum()
    row = {"VIX Regime": regime, "Days": n_days,
           "Mean VIX": vix_aligned[mask_t].mean()}
    for m in ["n_eff", "top5_mass", "herfindahl"]:
        g_mean = gcn_metrics[m][mask_t].mean()
        a_mean = gat_metrics[m][mask_t].mean()
        row[f"GCN {m}"] = g_mean
        row[f"GAT {m}"] = a_mean
    regime_rows.append(row)

regime_df = pd.DataFrame(regime_rows)
print("Concentration by VIX Regime:")
print(regime_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# ANOVA (does concentration vary across regimes?) and per-regime paired t-tests
print("ANOVA: does concentration vary across VIX regimes?\n")
for m in ["n_eff", "top5_mass", "herfindahl"]:
    for model_name, M in [("GCN", gcn_metrics), ("GAT", gat_metrics)]:
        groups = [M[m][regime_series == r].flatten() for r in ["High", "Mid", "Low"]]
        f, p = stats.f_oneway(*groups)
        print(f"  {model_name} {m:15s}: F={f:8.2f}, p={p:.2e}")

print("\nPaired t-tests GCN vs GAT within each regime:\n")
for regime in ["High", "Mid", "Low"]:
    mask_t = regime_series == regime
    print(f"  {regime} VIX ({mask_t.sum()} days):")
    for m in ["n_eff", "top5_mass", "herfindahl"]:
        g = gcn_metrics[m][mask_t].flatten()
        a = gat_metrics[m][mask_t].flatten()
        t_stat, p_val = stats.ttest_rel(g, a)
        diff = g.mean() - a.mean()
        print(f"    {m:15s}: diff(GCN-GAT)={diff:+.4f}, t={t_stat:7.2f}, p={p_val:.2e}")

In [ ]:
# Bar chart: regime concentration
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
regimes = ["High", "Mid", "Low"]
x = np.arange(len(regimes))
width = 0.35

for ax, m, label in zip(axes, ["n_eff", "top5_mass", "herfindahl"],
                              ["Effective neighbors", "Top-5 mass", "Herfindahl"]):
    g_means = [gcn_metrics[m][regime_series == r].mean() for r in regimes]
    a_means = [gat_metrics[m][regime_series == r].mean() for r in regimes]
    ax.bar(x - width/2, g_means, width, color=GCN_COLOR, label="GCN")
    ax.bar(x + width/2, a_means, width, color=GAT_COLOR, label="GAT")
    ax.set_xticks(x); ax.set_xticklabels(regimes)
    ax.set_xlabel("VIX Regime")
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend()

plt.suptitle("Concentration by VIX regime", fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Per-Sector Breakdown

In [ ]:
# Per-sector concentration: aggregate metrics over all windows, restricted to nodes in each sector
sector_rows = []
for sec, idx_list in sorted(sector_indices.items()):
    idx = np.array(idx_list)
    row = {"Sector": sec, "n_tickers": len(idx)}
    for m in ["n_eff", "top5_mass", "herfindahl"]:
        row[f"GCN {m}"] = gcn_metrics[m][:, idx].mean()
        row[f"GAT {m}"] = gat_metrics[m][:, idx].mean()
    sector_rows.append(row)

sector_df = pd.DataFrame(sector_rows).set_index("Sector")
print("Concentration by GICS Sector:")
print(sector_df.to_string(float_format="%.4f"))

In [ ]:
# Sector bar charts for 3 headline metrics
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, m, label in zip(axes, ["n_eff", "top5_mass", "herfindahl"],
                              ["Effective neighbors", "Top-5 mass", "Herfindahl"]):
    sectors_sorted = sector_df.index.tolist()
    x = np.arange(len(sectors_sorted))
    width = 0.35
    ax.bar(x - width/2, sector_df[f"GCN {m}"], width, color=GCN_COLOR, label="GCN")
    ax.bar(x + width/2, sector_df[f"GAT {m}"], width, color=GAT_COLOR, label="GAT")
    ax.set_xticks(x)
    ax.set_xticklabels(sectors_sorted, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel(label)
    ax.set_title(f"{label} by Sector")
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Per-Node (Per-Ticker) Breakdown

In [ ]:
# Per-ticker mean N_eff and top-5 mass (averaged across all windows)
node_df = pd.DataFrame({
    "Ticker": tickers,
    "Sector": sectors,
    "GCN N_eff": gcn_metrics["n_eff"].mean(axis=0),
    "GAT N_eff": gat_metrics["n_eff"].mean(axis=0),
    "GCN top5": gcn_metrics["top5_mass"].mean(axis=0),
    "GAT top5": gat_metrics["top5_mass"].mean(axis=0),
    "GCN H": gcn_metrics["herfindahl"].mean(axis=0),
    "GAT H": gat_metrics["herfindahl"].mean(axis=0),
})

# Most concentrated per model (lowest N_eff)
print("Most concentrated tickers by GCN (lowest N_eff):")
print(node_df.nsmallest(10, "GCN N_eff")[["Ticker", "Sector", "GCN N_eff", "GAT N_eff", "GCN top5", "GAT top5"]].to_string(index=False, float_format="%.3f"))

print("\nMost concentrated tickers by GAT (lowest N_eff):")
print(node_df.nsmallest(10, "GAT N_eff")[["Ticker", "Sector", "GCN N_eff", "GAT N_eff", "GCN top5", "GAT top5"]].to_string(index=False, float_format="%.3f"))

print("\nMost uniform tickers by GAT (highest N_eff):")
print(node_df.nlargest(10, "GAT N_eff")[["Ticker", "Sector", "GCN N_eff", "GAT N_eff", "GCN top5", "GAT top5"]].to_string(index=False, float_format="%.3f"))

In [ ]:
# Scatter: GCN N_eff vs GAT N_eff per ticker — do the two models agree on which tickers are concentrated?
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Use sector colours for legibility
sector_palette = {sec: plt.cm.tab20(i / max(len(unique_sectors) - 1, 1))
                  for i, sec in enumerate(unique_sectors)}

# Panel 1: N_eff
ax = axes[0]
for sec in unique_sectors:
    mask = node_df["Sector"] == sec
    ax.scatter(node_df.loc[mask, "GCN N_eff"], node_df.loc[mask, "GAT N_eff"],
               color=sector_palette[sec], label=sec, s=50, alpha=0.8, edgecolor="black", lw=0.3)
lo = min(node_df["GCN N_eff"].min(), node_df["GAT N_eff"].min())
hi = max(node_df["GCN N_eff"].max(), node_df["GAT N_eff"].max())
ax.plot([lo, hi], [lo, hi], color="black", ls="--", lw=0.8)
ax.set_xlabel("GCN N_eff (per ticker, mean over windows)")
ax.set_ylabel("GAT N_eff")
ax.set_title("Effective neighbors: GCN vs GAT per ticker")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))

r, p = stats.pearsonr(node_df["GCN N_eff"], node_df["GAT N_eff"])
ax.text(0.05, 0.95, f"r = {r:.3f}\np = {p:.2e}", transform=ax.transAxes,
        verticalalignment="top", bbox=dict(facecolor="white", alpha=0.8))

# Panel 2: top-5 mass
ax = axes[1]
for sec in unique_sectors:
    mask = node_df["Sector"] == sec
    ax.scatter(node_df.loc[mask, "GCN top5"], node_df.loc[mask, "GAT top5"],
               color=sector_palette[sec], label=sec, s=50, alpha=0.8, edgecolor="black", lw=0.3)
lo = min(node_df["GCN top5"].min(), node_df["GAT top5"].min())
hi = max(node_df["GCN top5"].max(), node_df["GAT top5"].max())
ax.plot([lo, hi], [lo, hi], color="black", ls="--", lw=0.8)
ax.set_xlabel("GCN top-5 mass")
ax.set_ylabel("GAT top-5 mass")
ax.set_title("Top-5 mass: GCN vs GAT per ticker")

r, p = stats.pearsonr(node_df["GCN top5"], node_df["GAT top5"])
ax.text(0.05, 0.95, f"r = {r:.3f}\np = {p:.2e}", transform=ax.transAxes,
        verticalalignment="top", bbox=dict(facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()

# Rank correlation (do the models agree on the ordering of tickers?)
print("\nRank correlations (Spearman):")
for m in [("GCN N_eff", "GAT N_eff"), ("GCN top5", "GAT top5"), ("GCN H", "GAT H")]:
    rho, p = stats.spearmanr(node_df[m[0]], node_df[m[1]])
    print(f"  {m[0]} vs {m[1]}: rho = {rho:+.3f}, p = {p:.2e}")

## 9. Interpretation

Read the results of Sections 3–8 against these questions:

**Headline question:**
- After restricting to the shared Pearson edge set and renormalizing, does GCN remain more concentrated than GAT (i.e. GCN N_eff < GAT N_eff, GCN top-5 mass > GAT top-5 mass, GCN Herfindahl > GAT Herfindahl)? If so, the 5b Gini finding was real — GAT genuinely spreads attention more uniformly. If the sign flips or shrinks to near-zero, the 5b finding was a sparsity artefact of Gini.

**Regime dependence (Section 6):**
- Does GCN–GAT concentration difference change across VIX regimes? If GCN concentrates more during stress (large Pearson correlations dominate), while GAT stays uniform, that supports the "GAT-as-regulariser" interpretation from 5b.

**Sector dependence (Section 7):**
- Are certain sectors systematically more concentrated in one model? Financials (many tickers, high within-sector correlations) may show different patterns than more dispersed sectors.

**Node agreement (Section 8):**
- Does the models' ranking of tickers-by-concentration agree (Spearman rho near +1) or diverge (rho near 0)? Disagreement would mean GAT is not merely a scaled version of Pearson — it identifies different tickers as having concentrated vs dispersed neighbour structure.

**Effect sizes matter more than p-values:**
With ~100k observations, even trivial differences are highly significant. Focus on Cohen's d in Section 3 and the visual gap in the histograms and CDFs (Section 4). A large Cohen's d (|d| > 0.5) with large raw differences is a real finding; a tiny Cohen's d with p < 1e-10 is statistical noise amplified by N.